# Effect of Microbe Derived Metabolines on Salmonella growth in a Mouse Gut

## Generate Constrained Models of the STM only condition and the Mixed STM condition

In [2]:
# Set the gurobi solver 

import os
from cobra.core.configuration import Configuration

os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

# Read in the gene weights CSV files

import pandas as pd

stm_threshold_10_90 = pd.read_csv("../imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("../imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("../imat_prep_test/stm_threshold_25_75.csv")

mixed_stm_threshold_10_90 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_10_90.csv")
mixed_stm_threshold_15_85 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_15_85.csv")
mixed_stm_threshold_25_75 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_25_75.csv")

# Make a python set of each gene weights CSV

stm_set_10_90 = set(stm_threshold_10_90["genes"].astype(str).str.strip())
stm_set_15_85 = set(stm_threshold_15_85["genes"].astype(str).str.strip())
stm_set_25_75 = set(stm_threshold_25_75["genes"].astype(str).str.strip())

mixed_stm_set_10_90 = set(mixed_stm_threshold_10_90["genes"].astype(str).str.strip())
mixed_stm_set_15_85 = set(mixed_stm_threshold_15_85["genes"].astype(str).str.strip())
mixed_stm_set_25_75 = set(mixed_stm_threshold_25_75["genes"].astype(str).str.strip())

# QC: Check the count of genes
print("STM1090 gene count",len(stm_set_10_90), "|", "STM1585 gene count", len(stm_set_15_85), "|", "STM2575 gene count", len(stm_set_25_75))
print("Mixed STM1090 gene count",len(mixed_stm_set_10_90), "|", "Mixed STM1585 gene count", len(mixed_stm_set_15_85), "|", "Mixed STM2575 gene count", len(mixed_stm_set_25_75))

# Load STM_V1_0

from cobra.io import load_model, load_json_model

control_model = load_model("STM_v1_0")

# Save all genes in the control model in a set

model_genes = set([g.id for g in control_model.genes])

# QC: Check the count of genes
print("STM_V1_0 Control Model has", len(model_genes), "total genes")

# Subset the gene weights lists for each sample to keep only the genes also found in the control model

stm1090_model = stm_threshold_10_90[stm_threshold_10_90["genes"].isin(model_genes)]
stm1585_model = stm_threshold_15_85[stm_threshold_15_85["genes"].isin(model_genes)]
stm2575_model = stm_threshold_25_75[stm_threshold_25_75["genes"].isin(model_genes)]

mixed_stm1090_model = mixed_stm_threshold_10_90[mixed_stm_threshold_10_90["genes"].isin(model_genes)]
mixed_stm1585_model = mixed_stm_threshold_15_85[mixed_stm_threshold_15_85["genes"].isin(model_genes)]
mixed_stm2575_model = mixed_stm_threshold_25_75[mixed_stm_threshold_25_75["genes"].isin(model_genes)]

# Align the gene weights lists to the control model genes list and assign missing genes a weight of 0

subset_weights_stm1090 = stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm1585 = stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm2575 = stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

subset_weights_mixedstm1090 = mixed_stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm1585 = mixed_stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm2575 = mixed_stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)


# QC: Confirm all samples have the same amount of gene weights

# 1. Make a list of the subset gene weights
subset_gene_weights = [
    subset_weights_stm1090,
    subset_weights_stm1585,
    subset_weights_stm2575,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
gene_counts = [len(weights) for weights in subset_gene_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(gene_counts)) == 1:
    print(f"All models have {gene_counts[0]} genes")
else:
    print("Gene counts differ between models")
    print(gene_counts)

# Convert gene weights to reaction weights with iMAT

from imatpy.parse_gpr import gene_to_rxn_weights

rxn_weights_stm1090 = gene_to_rxn_weights(control_model, subset_weights_stm1090)
rxn_weights_stm1585 = gene_to_rxn_weights(control_model, subset_weights_stm1585)
rxn_weights_stm2575 = gene_to_rxn_weights(control_model, subset_weights_stm2575)

rxn_weights_mixedstm1090 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1090)
rxn_weights_mixedstm1585 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1585)
rxn_weights_mixedstm2575 = gene_to_rxn_weights(control_model, subset_weights_mixedstm2575)

# QC: Confirm all samples have the same amount of reaction weights

# 1. Make a list of the reaction weights
reaction_weights = [
    rxn_weights_stm1090,
    rxn_weights_stm1585,
    rxn_weights_stm2575,
    rxn_weights_mixedstm1090,
    rxn_weights_mixedstm1585,
    rxn_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
reactions_counts = [len(weights) for weights in reaction_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(reactions_counts)) == 1:
    print(f"All models have {reactions_counts[0]} reactions")
else:
    print("Reaction counts differ between models")
    print(reactions_counts)

# Generate constrained models with iMAT by integrating the reaction weights

from imatpy.model_creation import generate_model

#stm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1090, method = "imat_restrictions")
#stm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1585, method = "imat_restrictions")
#stm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm2575, method = "imat_restrictions")

mixedstm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1090, method = "imat_restrictions")
mixedstm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1585, method = "imat_restrictions")
mixedstm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm2575, method = "imat_restrictions")

STM1090 gene count 4671 | STM1585 gene count 4671 | STM2575 gene count 4671
Mixed STM1090 gene count 4671 | Mixed STM1585 gene count 4671 | Mixed STM2575 gene count 4671
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu
STM_V1_0 Control Model has 1271 total genes
All models have 1271 genes
All models have 2545 reactions
Read LP format model from file /tmp/tmp079u6990.lp
Reading time = 0.01 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmp9ac3d0oc.lp
Reading time = 0.01 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpqe7odefh.lp
Reading time = 0.01 seconds
: 1802 rows, 5090 columns, 19612 nonzeros


In [3]:
# make a dictionary of all the models 

models = {
    "stm_unconstrained": control_model,
    "mixed_stm1090": mixedstm1090_model,
    "mixed_stm1585": mixedstm1585_model,
    "mixed_stm2575": mixedstm2575_model,
}

## Define the Mouse Minimal Medium Conditions

In [4]:
# Make a copy of the default medium from stm_v1_0
default_medium = dict(models["stm_unconstrained"].medium)

# Add mouse-derived, gut metabolites to the default medium to create a nouse gut minimal medium

mouse_min_medium = dict(default_medium)
mouse_min_medium.update({
    "EX_o2_e": 0.5, # updating to anoxic conditions
    "EX_no3_e": 0.5,
    "EX_for_e": 4.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_glc__D_e": 0.0 # removing from default medium
})

import pandas as pd
mouse_min_medium_df = pd.DataFrame.from_dict(mouse_min_medium, orient='index', columns=['value'])
mouse_min_medium_df

,value
EX_ca2_e,1000.00
EX_cbl1_e,0.01
EX_cl_e,1000.00
EX_co2_e,1000.00
EX_cobalt2_e,1000.00
EX_cu2_e,1000.00
EX_mg2_e,1000.00
EX_fe2_e,1000.00
EX_fe3_e,1000.00
EX_mn2_e,1000.00


## Figure out how to get how much of metabolite added is consumed and how much flux is going through reactions related to that metabolite - 09/11/26

Running `.optimize()` makes a solution object. The solution object has these attributes: objective value, status, fluxes, and shadow prices. The fluxes attribute is the flux in value and percent through a reaction using reaction id. 

It's only by inspecting the optimized model that I can get general input and output behavior such as information of metabolite uptake and secretion, what reaction it's in, flux, carbon number, and c - flux. The solved model is also used to inspect the nadh redox and main energy production and consumption. 

If I can't save the model, I need to output these data as outputs of the for loops so I can pull this up afterwards and compare across models. 

In [5]:
# copy a model
inspection_model = models["mixed_stm1090"].copy()

# attach a copy of the mouse_min_medium to the model
inspection_medium = dict(mouse_min_medium)
inspection_model.medium = inspection_medium

Read LP format model from file /tmp/tmp4lc61mgn.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros


In [6]:
inspection_model

Name,STM_v1_0
Memory address,5fd0ed5e810
Number of metabolites,1802
Number of reactions,2545
Number of genes,1271
Number of groups,0
Objective expression,1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4
Compartments,"cytosol, periplasm, extracellular space"


In [8]:
inspection_model.medium

{'EX_arab__L_e': 13.3,
 'EX_asp__L_e': 5.0,
 'EX_ca2_e': 1000.0,
 'EX_cbl1_e': 0.01,
 'EX_cl_e': 1000.0,
 'EX_co2_e': 1000.0,
 'EX_cobalt2_e': 1000.0,
 'EX_cu2_e': 1000.0,
 'EX_mg2_e': 1000.0,
 'EX_fe2_e': 1000.0,
 'EX_fe3_e': 1000.0,
 'EX_mn2_e': 1000.0,
 'EX_mobd_e': 1000.0,
 'EX_na1_e': 1000.0,
 'EX_nh4_e': 1000.0,
 'EX_for_e': 4.0,
 'EX_no3_e': 0.5,
 'EX_o2_e': 0.5,
 'EX_galct__D_e': 10.0,
 'EX_pi_e': 1000.0,
 'EX_zn2_e': 1000.0,
 'EX_pyr_e': 10.0,
 'EX_so4_e': 1000.0,
 'EX_glcr_e': 10.0,
 'EX_tungs_e': 1000.0,
 'EX_h2o_e': 100.0,
 'EX_h_e': 100.0,
 'EX_k_e': 1000.0,
 'EX_mal__L_e': 0.2}

In [6]:
# simulate growth of inspection_model
solution = inspection_model.optimize()

In [10]:
dir(solution)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_repr_html_',
 'fluxes',
 'get_primal_by_id',
 'objective_value',
 'reduced_costs',
 'shadow_prices',
 'status',
 'to_frame']

In [12]:
solution.objective_value

0.7706842505426165

In [19]:
solution.fluxes

3HAD180      0.020000
3HAD181      0.020000
3HAD40       0.170280
3HAD60       0.170280
3HAD80       0.170280
               ...   
12dgr2_ST    0.000072
OAL_ST       0.000722
OA4L_ST      0.000000
OA5L_ST      0.000722
OA4VL_ST     0.000000
Name: fluxes, Length: 2545, dtype: float64

In [18]:
inspection_model.metabolites.nadh_c.summary()

Percent,Flux,Reaction,Definition
0.07%,0.02,ACALD,acald_c + coa_c + nad_c <=> accoa_c + h_c + nadh_c
0.04%,0.01,AKGDH,akg_c + coa_c + nad_c --> co2_c + nadh_c + succoa_c
0.04%,0.01,ALCD19,glyald_c + h_c + nadh_c <=> glyc_c + nad_c
0.04%,0.01079,DKMPPD3,dkmpp_c + 3.0 h2o_c + 2.0 nad_c --> 2kmb_c + for_c + 4.0 h_c + 2.0 nadh_c + pi_c
71.48%,19.89,GAPD,g3p_c + nad_c + pi_c <=> 13dpg_c + h_c + nadh_c
0.14%,0.03853,GCALDD,gcald_c + h2o_c + nad_c --> glyclt_c + 2.0 h_c + nadh_c
0.04%,0.01,GLYCDx,glyc_c + nad_c --> dha_c + h_c + nadh_c
0.11%,0.03162,HACD8i,3hodcoa_c + nad_c --> 3ohodcoa_c + h_c + nadh_c
0.44%,0.1235,HISTD,h2o_c + histd_c + 2.0 nad_c --> 3.0 h_c + his__L_c + 2.0 nadh_c
0.73%,0.2042,IMPD,h2o_c + imp_c + nad_c --> h_c + nadh_c + xmp_c


## 09.13.26 Notes 
### What to add to `for` loops as output files
1. solution from `model.optimize()` with all the attributes: objective value, flux, status, and shadow prices. 
2. producing and consuming reactions after model.optimize() in summary methods. 
3. producing and consuming reactions before and after doing `model.optimize()` for nadhc, atpc, and metabolites tested: tetrathionate, succinate, 1,2ppd. 

### What I need to do 
- List how many result types I need to save 
- Find out the datatype for each object I want to save
- Find out how best to export that result. 

### How many experiments do I have? 
#### Design
mixedstm1090 + min media
mixedstm1585 + min media
mixedstm2575 + min media

mixedstm1090 + test media
mixedstm1585 + test media
mixedstm2575 + test media

#### Conditions
1. Succinate
2. Tetrathionate
3. Succinate and Tetrathionate
4. 1,2 PPD
5. Succinate and 1,2 PPD
6. Tetrathionate and 1,2 PPD
7. Succinate, Tetrathionate, and 1,2 PPD

### How many results do I need? 
#### Solution Object
6 solution objects per experiment condition. This will be used to compare flux between models grown on min media and models grown on test media in the same condition. 

#### Summary Method
6 model summaries per experimental condition. This will be used to compare the metabolite uptake and secretion of models grown on min media to models grown on the test media. 
6 model summaries of nadh per experimental conditon. This will be used to compare nadh production and consumption of models grown on min media to models grown on the test media. 
6 model summaries of atpc per experimental condition. This will be used to compare energy production and consumption in models grown on min media to models grown on the test media. 
6 model summaries of the test metabolite(s) per experimental condition. This will be used to compare production and consumption of that metabolite in the min media and test media. 

### Exploring the solution object

In [ ]:
solution = inspection_model.optimize()

In [13]:
solution

,fluxes,reduced_costs
3HAD180,0.020000,NaN
3HAD181,0.020000,NaN
3HAD40,0.170280,NaN
3HAD60,0.170280,NaN
3HAD80,0.170280,NaN
...,...,...
12dgr2_ST,0.000072,NaN
OAL_ST,0.000722,NaN
OA4L_ST,0.000000,NaN
OA5L_ST,0.000722,NaN


In [7]:
type(solution)

cobra.core.solution.Solution

In [16]:
dir(solution)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_repr_html_',
 'fluxes',
 'get_primal_by_id',
 'objective_value',
 'reduced_costs',
 'shadow_prices',
 'status',
 'to_frame']

In [11]:
from cobra.core import get_solution

real_solution = get_solution(inspection_model)

In [14]:
type(real_solution)

cobra.core.solution.Solution

In [15]:
dir(real_solution)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_repr_html_',
 'fluxes',
 'get_primal_by_id',
 'objective_value',
 'reduced_costs',
 'shadow_prices',
 'status',
 'to_frame']

In [20]:
assert real_solution == solution, "The two solutions are not equal"

AssertionError: The two solutions are not equal

In [21]:
assert type(real_solution) == type(solution), "The two solutions are not equal"

In [24]:
assert real_solution.fluxes.ABUTtex == solution.fluxes.ABUTtex, "The two solutions are not equal"

## How do I save what I need from the solution object? 

In [43]:
solution.fluxes.to_frame()

,fluxes
3HAD180,0.020000
3HAD181,0.020000
3HAD40,0.170280
3HAD60,0.170280
3HAD80,0.170280
...,...
12dgr2_ST,0.000072
OAL_ST,0.000722
OA4L_ST,0.000000
OA5L_ST,0.000722


In [ ]:
solution

<bound method Solution.to_frame of <Solution 0.771 at 0xb3c8d0699d0>>

## Exploring the Model.Summary object

In [48]:
summary = inspection_model.summary()

print(summary)

Metabolite,Reaction,Flux,C-Number,C-Flux
arab__L_e,EX_arab__L_e,13.3,5,28.02%
asp__L_e,EX_asp__L_e,5,4,8.43%
ca2_e,EX_ca2_e,0.003651,0,0.00%
cl_e,EX_cl_e,0.003651,0,0.00%
cobalt2_e,EX_cobalt2_e,0.002434,0,0.00%
cu2_e,EX_cu2_e,0.002434,0,0.00%
fe2_e,EX_fe2_e,0.01673,0,0.00%
galct__D_e,EX_galct__D_e,10,6,25.28%
glcr_e,EX_glcr_e,10,6,25.28%
h2o_e,EX_h2o_e,8.488,0,0.00%


In [ ]:
type(summary)

cobra.summary.model_summary.ModelSummary

In [ ]:
import pandas as pd
test_uptake = summary.uptake_flux.to_csv("uptake_flux.csv")
test_secretion = summary.secretion_flux.to_csv("secretion_flux.csv")
test_all = summary.to_frame().to_csv("summary_flux.csv")
full_summary_df = pd.concat(
    [summary.uptake_flux.assign(type='uptake'),
     summary.secretion_flux.assign(type='secretion')
     ],
     ignore_index=True
)

full_summary_df["objective_value"] = pd.NA
full_summary_df.iloc[0, full_summary_df.columns.get_loc("objective_value")] = solution.objective_value
full_summary_df

# if i want to report carbon flux, I need to write a afunction to do this myself because it is not stored
# in the summary object. 

,flux,reaction,metabolite,type,objective_value
0,13.300000,EX_arab__L_e,arab__L_e,uptake,0.770684
1,5.000000,EX_asp__L_e,asp__L_e,uptake,<NA>
2,0.003651,EX_ca2_e,ca2_e,uptake,<NA>
3,0.003651,EX_cl_e,cl_e,uptake,<NA>
4,0.002434,EX_cobalt2_e,cobalt2_e,uptake,<NA>
...,...,...,...,...,...
340,0.000000,EX_xan_e,xan_e,secretion,<NA>
341,0.000000,EX_xmp_e,xmp_e,secretion,<NA>
342,0.000000,EX_xtsn_e,xtsn_e,secretion,<NA>
343,0.000000,EX_xyl__D_e,xyl__D_e,secretion,<NA>


## Getting the Summaries for each metabolite added, nadh, and atpc

In [ ]:
atp_summary = inspection_model.metabolites.nad
atp_summary

Percent,Flux,Reaction,Definition
53.35%,65.5,ACKr,ac_c + atp_c <=> actp_c + adp_c
0.01%,0.01,CYTK2,atp_c + dcmp_c <=> adp_c + dcdp_c
0.01%,0.01,DGK1,atp_c + dgmp_c <=> adp_c + dgdp_c
0.01%,0.01,NDPK5,atp_c + dgdp_c <=> adp_c + dgtp_c
0.03%,0.04014,NDPK6,atp_c + dudp_c <=> adp_c + dutp_c
0.01%,0.01,NDPK7,atp_c + dcdp_c <=> adp_c + dctp_c
16.20%,19.89,PGK,3pg_c + atp_c <=> 13dpg_c + adp_c
0.01%,0.01,PPAKr,adp_c + ppap_c <=> atp_c + ppa_c
30.35%,37.26,PYK,adp_c + h_c + pep_c --> atp_c + pyr_c
0.03%,0.04014,URIDK2r,atp_c + dump_c <=> adp_c + dudp_c


In [10]:
atp_summary.producing_flux

,flux,reaction,percent
ACKr,65.500608,ACKr,0.533467
AP5AH,0.000000,AP5AH,0.000000
CYTK2,0.010000,CYTK2,0.000081
DGK1,0.010000,DGK1,0.000081
NDPK5,0.010000,NDPK5,0.000081
NDPK6,0.040143,NDPK6,0.000327
NDPK7,0.010000,NDPK7,0.000081
PGK,19.893448,PGK,0.162021
PPAKr,0.010000,PPAKr,0.000081
PYK,37.258596,PYK,0.303451


## Exploring Reaction Fluxes

In [11]:
reactions_summary = inspection_model.reactions.summary()

AttributeError: DictList has no attribute or entry summary

## Simulate mixed_stm growth with mouse minimal medium

### Add tetrathionate to the mouse gut minimal media and simulate growth

In [11]:
tet_growth = []

for name, model in models.copy().items():

    min_media = dict(mouse_min_medium)
    tet_min_media = dict(mouse_min_medium)
    tet_min_media.update({
        "EX_tet_e": 4.0
    })

    min_model = model.copy()
    min_model.medium = min_media

    tet_model = model.copy()
    tet_model.medium = tet_min_media

    min_solution = min_model.optimize()
    tet_solution = tet_model.optimize()

    tet_growth.append({
        "Model": name,
        "Tet Minimal Media": tet_solution.objective_value,
        "Minimal Media": min_solution.objective_value,
        "Tet Flux": tet_solution.fluxes["EX_tet_e"]
    })
     
tet_growth_df = pd.DataFrame(tet_growth)
tet_growth_df

Read LP format model from file /tmp/tmpkqzoolzr.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros


Read LP format model from file /tmp/tmpdb8diivo.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpab5bxx8p.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpnf7wt8cg.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros

Interrupt request received
Read LP format model from file /tmp/tmpg2tqmvlt.lp
Reading time = 0.02 seconds
: 3179 rows, 6432 columns, 25082 nonzeros
Read LP format model from file /tmp/tmp2nyicyml.lp
Reading time = 0.02 seconds
: 3179 rows, 6432 columns, 25082 nonzeros
Read LP format model from file /tmp/tmp_67mrc5s.lp
Reading time = 0.03 seconds
: 3901 rows, 7068 columns, 27884 nonzeros
Read LP format model from file /tmp/tmpar_5eurg.lp
Reading time = 0.02 seconds
: 3901 rows, 7068 columns, 27884 nonzeros


,Model,Tet Minimal Media,Minimal Media,Tet Flux
0,stm_unconstrained,0.891746,0.809029,-4.0
1,mixed_stm1090,0.860537,0.770684,-4.0
2,mixed_stm1585,0.855216,0.767465,-4.0
3,mixed_stm2575,0.844297,0.754592,-4.0


In [12]:
succ_growth = []

for name, model in models.copy().items():

    min_media = dict(mouse_min_medium)
    succ_min_media = dict(mouse_min_medium)
    succ_min_media.update({
        "EX_succ_e": 3.0
    })

    min_model = model.copy()
    min_model.medium = min_media

    succ_model = model.copy()
    succ_model.medium = succ_min_media

    min_solution = min_model.optimize()
    succ_solution = succ_model.optimize()

    succ_growth.append({
        "Model": name,
        "Succ Minimal Media": succ_solution.objective_value,
        "Minimal Media": min_solution.objective_value,
        "Succ Flux": succ_solution.fluxes["EX_succ_e"]
    })
     
succ_growth_df = pd.DataFrame(succ_growth)
succ_growth_df

Read LP format model from file /tmp/tmpnwpwh5jm.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpofghksxr.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmphoiyv318.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpk3hfe_nt.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpzum1phh6.lp
Reading time = 0.02 seconds
: 3179 rows, 6432 columns, 25082 nonzeros
Read LP format model from file /tmp/tmp5q9g560w.lp
Reading time = 0.02 seconds
: 3179 rows, 6432 columns, 25082 nonzeros
Read LP format model from file /tmp/tmpj3izt658.lp
Reading time = 0.02 seconds
: 3901 rows, 7068 columns, 27884 nonzeros
Read LP format model from file /tmp/tmpotv74d06.lp
Reading time = 0.02 seconds
: 3901 rows, 7068 columns, 27884 nonzeros


,Model,Succ Minimal Media,Minimal Media,Succ Flux
0,stm_unconstrained,0.809029,0.809029,-2.391792
1,mixed_stm1090,0.771078,0.770684,-0.050000
2,mixed_stm1585,0.767909,0.767465,-0.050000
3,mixed_stm2575,0.755038,0.754592,-1.914974


In [13]:
tet_and_succ_growth = []

for name, model in models.copy().items():

    min_media = dict(mouse_min_medium)
    tet_and_succ_min_media = dict(mouse_min_medium)
    tet_and_succ_min_media.update({
        "EX_tet_e": 4.0,
        "EX_succ_e": 3.0
    })

    min_model = model.copy()
    min_model.medium = min_media

    tet_and_succ_model = model.copy()
    tet_and_succ_model.medium = tet_and_succ_min_media

    min_solution = min_model.optimize()
    tet_and_succ_solution = tet_and_succ_model.optimize()

    tet_and_succ_growth.append({
        "Model": name,
        "Tet and Succ Minimal Media": tet_and_succ_solution.objective_value,
        "Minimal Media": min_solution.objective_value,
        "Tet Flux": tet_and_succ_solution.fluxes["EX_tet_e"],
        "Succ Flux": tet_and_succ_solution.fluxes["EX_succ_e"]
    })
     
tet_and_succ_growth_df = pd.DataFrame(tet_and_succ_growth)
tet_and_succ_growth_df

Read LP format model from file /tmp/tmpnxbfc0yf.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpgup8eq0v.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpdhpjg88p.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp8av6ybml.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp0smwo1kw.lp
Reading time = 0.02 seconds
: 3179 rows, 6432 columns, 25082 nonzeros
Read LP format model from file /tmp/tmpprdk71tz.lp
Reading time = 0.02 seconds
: 3179 rows, 6432 columns, 25082 nonzeros
Read LP format model from file /tmp/tmpo5dqugb2.lp
Reading time = 0.02 seconds
: 3901 rows, 7068 columns, 27884 nonzeros
Read LP format model from file /tmp/tmphppxxlzj.lp
Reading time = 0.02 seconds
: 3901 rows, 7068 columns, 27884 nonzeros


,Model,Tet and Succ Minimal Media,Minimal Media,Tet Flux,Succ Flux
0,stm_unconstrained,0.891746,0.809029,-4.0,0.000000
1,mixed_stm1090,0.860982,0.770684,-4.0,-2.938365
2,mixed_stm1585,0.855663,0.767465,-4.0,-2.949905
3,mixed_stm2575,0.844744,0.754592,-4.0,-0.050000


In [ ]:
ppd_growth = []

for name, model in models.copy().items():

    min_media = dict(mouse_min_medium)
    ppd_min_media = dict(mouse_min_medium)
    ppd_min_media.update({
        "EX_12ppd__R_e": 0.1,
        "EX_12ppd__S_e": 0.1
    })

    min_model = model.copy()
    min_model.medium = min_media

    ppd_model = model.copy()
    ppd_model.medium = ppd_min_media

    min_solution = min_model.optimize()
    ppd_solution = ppd_model.optimize()

    ppd_growth.append({
        "Model": name,
        "PPD Growth": ppd_solution.objective_value,
        "Minimal Media": min_solution.objective_value,
        "1,2 PPD R Flux": ppd_solution.fluxes["EX_12ppd__R_e"],
        "1,2 PPD S Flux": ppd_solution.fluxes["EX_12ppd__S_e"]
    })
     
ppd_growth_df = pd.DataFrame(ppd_growth)
ppd_growth_df

<Solution 0.863 at 0x11330f894a90>
<Solution 0.771 at 0x11330f894f10>
<Solution 0.892 at 0x11330f992f50>
<Solution 0.892 at 0x1133143c9110>


In [14]:
tet_succ_ppd_growth = []

for name, model in models.copy().items():

    min_media = dict(mouse_min_medium)
    tet_succ_ppd_min_media = dict(mouse_min_medium)
    tet_succ_ppd_min_media.update({
        "EX_succ_e": 3.0,
        "EX_tet_e": 4.0,
        "EX_12ppd__R_e": 0.1,
        "EX_12ppd__S_e": 0.1
    })

    min_model = model.copy()
    min_model.medium = min_media

    tet_succ_ppd_model = model.copy()
    tet_succ_ppd_model.medium = tet_succ_ppd_min_media

    min_solution = min_model.optimize()
    tet_succ_ppd_solution = tet_succ_ppd_model.optimize()

    tet_succ_ppd_growth.append({
        "Model": name,
        "Tet-Succ-PPD Growth": tet_succ_ppd_solution.objective_value,
        "Minimal Media": min_solution.objective_value,
        "Tet Flux": tet_succ_ppd_solution.fluxes["EX_tet_e"],
        "Succ Flux": tet_succ_ppd_solution.fluxes["EX_succ_e"],
        "1,2 PPD R Flux": tet_succ_ppd_solution.fluxes["EX_12ppd__R_e"],
        "1,2 PPD S Flux": tet_succ_ppd_solution.fluxes["EX_12ppd__S_e"]
    })
     
tet_succ_ppd_growth_df = pd.DataFrame(tet_succ_ppd_growth)
tet_succ_ppd_growth_df

Read LP format model from file /tmp/tmpaz690054.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpge5ewpm_.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpv8vn1szd.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp26n67bql.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp4d36e_xq.lp
Reading time = 0.02 seconds
: 3179 rows, 6432 columns, 25082 nonzeros
Read LP format model from file /tmp/tmpygofztdp.lp
Reading time = 0.02 seconds
: 3179 rows, 6432 columns, 25082 nonzeros
Read LP format model from file /tmp/tmp_jqn5qa0.lp
Reading time = 0.02 seconds
: 3901 rows, 7068 columns, 27884 nonzeros
Read LP format model from file /tmp/tmpmfb49z1e.lp
Reading time = 0.03 seconds
: 3901 rows, 7068 columns, 27884 nonzeros


,Model,Tet-Succ-PPD Growth,Minimal Media,Tet Flux,Succ Flux,"1,2 PPD R Flux","1,2 PPD S Flux"
0,stm_unconstrained,0.891746,0.809029,-4.0,0.00,0.00,0.0
1,mixed_stm1090,0.863035,0.770684,-4.0,-0.05,-0.10,-0.1
2,mixed_stm1585,0.857573,0.767465,-4.0,-0.05,-0.01,-0.1
3,mixed_stm2575,0.846087,0.754592,-4.0,-0.05,-0.01,-0.1
